In [1]:
import polars as pl
import pandas as pd
import numpy as np
import os
import glob
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")

print(">>> 正在自动搜寻数据文件路径...")
possible_paths = glob.glob("/kaggle/input/**/player_play.csv", recursive=True) + \
                 glob.glob("./**/player_play.csv", recursive=True)

if possible_paths:
    DATA_DIR = os.path.dirname(possible_paths[0])
    print(f"✅ 成功找到数据目录: {DATA_DIR}")
else:
    print("❌ 警告: 未找到数据，请确保已 Add Data 挂载。默认路径设为 './'")
    DATA_DIR = "./"

>>> 正在自动搜寻数据文件路径...
✅ 成功找到数据目录: /kaggle/input/datasets/yqx123456/12345678/nfl-big-data-bowl-2027


In [2]:
print(">>> 1. 提取 Combine 物理运动学指纹 (方案A)...")
# 读取追踪数据，过滤掉球，只保留球员
ct = pl.read_csv(os.path.join(DATA_DIR, "combine_tracking.csv")).filter(pl.col("entity_type") == "PLAYER")

# 单指数冲刺模型 v(t) = v_max * (1 - exp(-t/tau))
def sprint_model(t, v_max, tau): return v_max * (1 - np.exp(-t / tau))

def extract_40yd(df):
    df = df.sort("time")
    if len(df) < 25: return None
    t = np.arange(len(df)) * 0.1 # 10Hz 采样
    s = df["s"].to_numpy()
    if np.max(s) < 2.0: return None
    try:
        popt, _ = curve_fit(sprint_model, t, s, p0=[10.0, 0.5], bounds=([5.0, 0.1], [15.0, 2.0]))
        return {"40_v_max": popt[0], "40_tau": popt[1], "40_a_max": popt[0] / popt[1]}
    except: return None

def extract_agility(df):
    df = df.sort("time")
    if len(df) < 25: return None
    s = df["s"].to_numpy()
    dir_rad = np.radians(df["dir"].to_numpy())
    w = 7 if len(dir_rad) > 7 else len(dir_rad) - (1 - len(dir_rad) % 2)
    if w < 5: return None
    # 平滑方向角，计算角速度（反映髋部翻转能力）
    dir_smooth = savgol_filter(dir_rad, window_length=w, polyorder=2)
    ang_vel = np.abs(np.diff(dir_smooth) / 0.1)
    return {"peak_speed": np.max(s), "max_angular_vel": np.max(ang_vel) if len(ang_vel)>0 else 0}

results_40 = []
results_ag = []

for keys, df in ct.group_by(["nfl_id", "drill_type", "event_id"]):
    nfl_id, drill_type, _ = keys
    if drill_type == "FORTY_YARD_DASH":
        feats = extract_40yd(df)
        if feats: feats["nfl_id"] = nfl_id; results_40.append(feats)
    elif drill_type.startswith("SKILL_DRILLS") or drill_type in ["THREE_CONE_DRILL", "SHORT_SHUTTLE"]:
        feats = extract_agility(df)
        if feats: feats.update({"nfl_id": nfl_id, "drill_type": drill_type}); results_ag.append(feats)

df_40 = pl.DataFrame(results_40).group_by("nfl_id").agg([pl.col("40_v_max").median(), pl.col("40_tau").median(), pl.col("40_a_max").median()])
df_ag = pl.DataFrame(results_ag).group_by(["nfl_id", "drill_type"]).agg(pl.col("max_angular_vel").median())
df_ag_pivot = df_ag.pivot(values="max_angular_vel", index="nfl_id", columns="drill_type")
df_ag_pivot.columns = [f"ang_vel_{c}" if c != "nfl_id" else c for c in df_ag_pivot.columns]
print(f"✅ 联合测试指纹提取完成。")

>>> 1. 提取 Combine 物理运动学指纹 (方案A)...
✅ 联合测试指纹提取完成。


In [3]:
print(">>> 2. 提取比赛轨迹特征并拼装基础数据 (方案B)...")
# 提取比赛追踪特征（单回合峰值速度/加速度）
gt = pl.scan_csv(os.path.join(DATA_DIR, "game_tracking_2023.csv")).select(["game_id", "play_id", "nfl_id", "s", "a"])
gt_feats = (
    gt.group_by(["game_id", "play_id", "nfl_id"])
      .agg([
          pl.col("s").max().alias("game_peak_speed"),
          pl.col("s").mean().alias("game_mean_speed"),
          pl.col("a").max().alias("game_max_accel")
      ]).collect()
)

# 加载基础文件（处理 NA 报错）
combine_res = pl.read_csv(os.path.join(DATA_DIR, "combine_results.csv"), null_values=["NA", ""])
pp = pl.read_csv(os.path.join(DATA_DIR, "player_play.csv"), null_values=["NA", ""], infer_schema_length=10000)

# 提取含有 Target 的回合
target_df = (
    pp.filter(pl.col("separation_at_pass_forward").is_not_null())
      .select(["game_id", "play_id", "nfl_id", "separation_at_pass_forward", 
               "route_ran", "team_coverage_man_zone", "team_coverage_type", "cushion"])
)

# 拼装静态特征大表（目标 + 轨迹 + 体测 + 物理指纹）
base_df = (
    target_df
    .join(gt_feats, on=["game_id", "play_id", "nfl_id"], how="left")
    .join(combine_res.drop(["draft_year", "combine_position"]), on="nfl_id", how="left")
    .join(df_40, on="nfl_id", how="left")
    .join(df_ag_pivot, on="nfl_id", how="left")
)

# 转为 Pandas 给 LightGBM 使用
base_pd = base_df.to_pandas().fillna(0)
print(f"✅ 基础特征拼装完成，数据维度: {base_pd.shape}")

>>> 2. 提取比赛轨迹特征并拼装基础数据 (方案B)...
✅ 基础特征拼装完成，数据维度: (46737, 37)


In [4]:
print(">>> 3. 加入战术上下文并训练 WR 模型 (方案F)...")
# One-Hot 编码战术特征
context_pd = pd.get_dummies(base_pd, columns=["route_ran", "team_coverage_man_zone", "team_coverage_type"], drop_first=False)

# 读取球员位置信息
players = pd.read_csv(os.path.join(DATA_DIR, "players.csv"), na_values=["NA"])
merged = pd.merge(context_pd, players[["nfl_id", "nfl_position"]], on="nfl_id", how="inner")

# 筛选 WR
wr_df = merged[merged["nfl_position"] == "WR"].copy()
target_col = "separation_at_pass_forward"
drop_cols = [target_col, "game_id", "play_id", "nfl_id", "nfl_position"]
feature_cols = [c for c in wr_df.columns if c not in drop_cols]

X = wr_df[feature_cols]
y = wr_df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 基线特征（传统指标）
base_feats = ["forty", "three_cone", "short_shuttle", "vertical", "broad_jump", "ngs_final_score"]
base_feats = [c for c in base_feats if c in X.columns]

# 训练基线模型
model_base = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=7, random_state=42, verbose=-1)
model_base.fit(X_train[base_feats], y_train)
mae_base = mean_absolute_error(y_test, model_base.predict(X_test[base_feats]))
r2_base = r2_score(y_test, model_base.predict(X_test[base_feats]))

# 训练增强模型
model_aug = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=7, random_state=42, verbose=-1)
model_aug.fit(X_train, y_train)
mae_aug = mean_absolute_error(y_test, model_aug.predict(X_test))
r2_aug = r2_score(y_test, model_aug.predict(X_test))

print("\n" + "="*40)
print(f"基线模型 MAE: {mae_base:.4f} | R²: {r2_base:.4f}")
print(f"增强模型 MAE: {mae_aug:.4f} | R²: {r2_aug:.4f}")
print(f"🔥 误差降低幅度: {(mae_base - mae_aug) / mae_base * 100:.2f}%")
print("="*40)

# 输出 SHAP 图
explainer = shap.TreeExplainer(model_aug)
shap_values = explainer.shap_values(X_test.iloc[:1000])
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test.iloc[:1000], show=False, max_display=15)
plt.title("SHAP Summary - WR (Sensor-Augmented Model)", fontsize=14)
plt.tight_layout()
plt.savefig("shap_WR.png", dpi=300, bbox_inches='tight')
plt.close()
print("✅ SHAP 图已保存为 shap_WR.png")

>>> 3. 加入战术上下文并训练 WR 模型 (方案F)...

基线模型 MAE: 1.6563 | R²: 0.0193
增强模型 MAE: 1.3436 | R²: 0.2875
🔥 误差降低幅度: 18.88%
✅ SHAP 图已保存为 shap_WR.png


In [5]:
print(">>> 4. 寻找隐藏人才与速度陷阱...")
pred_all = model_aug.predict(X)
wr_df["actual_sep"] = y
wr_df["pred_sep"] = pred_all
wr_df["residual"] = wr_df["actual_sep"] - wr_df["pred_sep"]

# 聚合到球员级别
player_perf = wr_df.groupby("nfl_id").agg({
    "actual_sep": "mean", "pred_sep": "mean", "residual": "mean",
    "forty": "first", "40_v_max": "first", "play_id": "count"
}).rename(columns={"play_id": "targets"})

# 过滤样本过少的球员
player_perf = player_perf[player_perf["targets"] >= 20].reset_index()
player_perf["nfl_id"] = player_perf["nfl_id"].astype(int)
players["nfl_id"] = players["nfl_id"].astype(int)
player_perf = player_perf.merge(players[["nfl_id", "display_name"]], on="nfl_id", how="left")

# 获取名单
hidden_gems = player_perf.sort_values("residual", ascending=False).head(10)
overrated = player_perf.sort_values("residual", ascending=True).head(10)

print("\n🌟 隐藏人才名单 (Hidden Gems):")
print(hidden_gems[["display_name", "targets", "actual_sep", "pred_sep", "residual", "forty", "40_v_max"]].to_string(index=False))
print("\n💦 速度陷阱名单 (Overvalued):")
print(overrated[["display_name", "targets", "actual_sep", "pred_sep", "residual", "forty", "40_v_max"]].to_string(index=False))

>>> 4. 寻找隐藏人才与速度陷阱...

🌟 隐藏人才名单 (Hidden Gems):
      display_name  targets  actual_sep  pred_sep  residual  forty  40_v_max
    Jacob Copeland       25    3.612275  3.310447  0.301828   4.42 12.164597
       Jalen Wayne       37    3.035115  2.746427  0.288689   4.51  0.000000
   Tahj Washington       67    3.817335  3.544787  0.272547   0.00  0.000000
   Ja'Corey Brooks       46    3.337297  3.157554  0.179743   0.00  0.000000
       Arian Smith      227    3.418554  3.249909  0.168645   4.36 12.698322
      Jalen Royals       41    3.751069  3.585213  0.165855   4.42 12.431655
Bryce Ford-Wheaton       46    2.790224  2.631675  0.158549   4.38 11.903889
      Isaiah Neyor       24    2.934634  2.790288  0.144346   4.40 12.000425
     Jaray Jenkins       29    3.008062  2.882112  0.125950   4.60 11.344511
        Tai Felton       46    3.355164  3.247684  0.107480   4.37 11.988016

💦 速度陷阱名单 (Overvalued):
   display_name  targets  actual_sep  pred_sep  residual  forty  40_v_max
   Donta

In [6]:
print(">>> 5. 生成最终可视化图表...")
# 图 1：性能对比图
fig, ax1 = plt.subplots(figsize=(8, 5))
x = np.arange(2)
width = 0.35
rects1 = ax1.bar(x - width/2, [r2_base, r2_aug], width, label='R² Score', color='#4C72B0')
ax1.set_ylabel('R² Score (Higher is Better)', color='#4C72B0')
ax1.set_ylim(0, 0.35)
ax1.set_xticks(x)
ax1.set_xticklabels(['Traditional\nBaseline', 'Sensor-Augmented\nOurs'])
ax1.tick_params(axis='y', labelcolor='#4C72B0')

ax2 = ax1.twinx()
rects2 = ax2.bar(x + width/2, [mae_base, mae_aug], width, label='MAE (Lower is Better)', color='#DD8452')
ax2.set_ylabel('MAE (Yards)', color='#DD8452')
ax2.set_ylim(1.0, 1.8)
ax2.tick_params(axis='y', labelcolor='#DD8452')
plt.title('Model Performance: Baseline vs Sensor-Augmented', pad=15)
plt.tight_layout()
plt.savefig("performance_comparison.png", dpi=300, bbox_inches='tight')
plt.close()

# 图 2：隐藏人才散点图（过滤掉四十码缺失为0的）
plot_df = player_perf[player_perf['forty'] > 0].copy()
gems = plot_df.sort_values("residual", ascending=False).head(5)
traps = plot_df.sort_values("residual", ascending=True).head(5)

plt.figure(figsize=(10, 6))
plt.scatter(plot_df['forty'], plot_df['residual'], alpha=0.4, color='gray', label='Average Players')
plt.scatter(gems['forty'], gems['residual'], color='#FF4B4B', s=80, label='Hidden Gems (Undervalued)', zorder=5)
plt.scatter(traps['forty'], traps['residual'], color='#4B8BFF', s=80, label='Speed Traps (Overvalued)', zorder=5)

for i, row in gems.iterrows():
    plt.annotate(row['display_name'], (row['forty'], row['residual']), xytext=(8, 5), textcoords='offset points', color='#D32F2F', fontsize=9, fontweight='bold')
for i, row in traps.iterrows():
    plt.annotate(row['display_name'], (row['forty'], row['residual']), xytext=(8, -10), textcoords='offset points', color='#1976D2', fontsize=9, fontweight='bold')

plt.axhline(0, color='black', linestyle='--', linewidth=1.5)
plt.xlabel('Traditional 40-Yard Dash Time (s)', fontsize=12)
plt.ylabel('Separation Residual (Actual - Predicted)', fontsize=12)
plt.title('Uncovering Hidden Talents: 40-Yard Dash vs. On-Field Separation', fontsize=14, pad=15)
plt.legend(loc='lower left')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("hidden_talents_scatter.png", dpi=300, bbox_inches='tight')
plt.close()

print("✅ 所有图表已保存生成 (performance_comparison.png, hidden_talents_scatter.png, shap_WR.png)")

>>> 5. 生成最终可视化图表...
✅ 所有图表已保存生成 (performance_comparison.png, hidden_talents_scatter.png, shap_WR.png)
